# Welcome to the MTR blog post Jupyter Notebook!

If this is your first time running a Juptyer Notebook, there's a lot of tutorials available online to help. [Here's one](https://www.dataquest.io/blog/jupyter-notebook-tutorial/) for your convenience.

## Introduction

This notebook contains everything needed to reproduce the Inversion Recovery T<sub>1</sub> blog post on the [qMRLab website](). In fact, this notebook generated the HTML for the blog post too! This notebook is currently running on a MyBinder server that only you can access, but if you want to be kept up-to-date on any changes that the developpers make to this notebook, you should go to it's [GitHub repository](https://github.com/qMRLab/t1_notebooks) and follow it by clicking the "Watch" button in the top right (you may need to create a GitHub account, if you don't have one already).

## Tips

Here's a few things you can do in this notebook

### Code
* Run the entire processing by clicking above on the "Kernel" tab, then "Restart & Run All". Currently, the processing time is approximately **30-45 minutes**. It will be complete when none of the cells have an asterix "\*" in the square brackets.
* To change the code, you need to click once on code cells. To re-run that cell, click the "Run" button above when the cell is selected.
  * **Note:** Cells can depend on previous cells, or even on previous runs of the cell itself, so it's best to run all the previous cells beforehand.
* This binder runs on SoS, which allows the mixing of Octave (i.e. an open-source MATLAB) and Python cells. Take a look a the drop down menu on the top right of the cells to know which one you are running.
* To transfer data from cells of one language to another, you need to create a new cell in the incoming language and run `%get (param name) --from (outgoing language)`. See cells below for several examples within this notebook.

### HTML
* To reproduce the HTML of the blog post, run the entire processing pipeline (see point one in the previous section), then save the notebook (save icon, top left). Now, click on the drop down menu on the left pannel, and select `%sossave --to html --force` . After a few seconds, it should output "Workflow saved to InversionRecovery.html" – click on the HTML name, and you're done!
* Cells with tags called "scratch" are not displayed in the generated HTML.
* Cells with the tag "report_output" display the output (e.g. figures) in the generated HTML.
* Currently in an un-run notebook, the HTML is not formatted like the website. To do so, run the Python module import cell (`# Module imports`) and then very last cell (`display(HTML(...`).

**If you have any other questions or comments, please raise them in a [GitHub issue](https://github.com/qMRLab/t1_notebooks/issues).**

# Note

The following cell is meant to be displayed for instructional purposes in the blog post HTML when "All cells" gets displayed (i.e. the Octave code).

In [ ]:
%use python3
# PYTHON CODE
# Module imports

import matplotlib.pyplot as plt
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML

<center><h1 style="font-family: timesnewroman;font-size: 40px;">MT Series: Magnetization Transfer Ratio (MTR)</h1></center>
<p>

<div class=blog_body>
<p style="text-align:justify;">
Conventional MRI techniques, such as those used for clinical diagnosis, can only directly measure hydrogen bonded to water molecules. This means that there’s a large proportion of body mass not visible with clinical MRIs, such as non-hydrogren atoms (different resonance frequencies) and hydrogen atoms bonded to large molecules which restricts the motion of the atoms (rapid signal decay, T2 ~ μs). The latter, called  macromolecules, play an important role in the physiology of the body; for example, myelin in the white matter of the brain plays a major role in signal conduction, and is composed largely of macromolecules (lipids and proteins). Although the images acquired by clinical MRI machines can only be generated from signal from mobile hydrogen, these interact with nearby molecules and atoms via the electromagnetic fields they mutually generate, and in the 70s and 80s a cross-relaxation mechanism was discovered that sensitizes mobile protons to nearby targetted semi-solid molecules, such as myelin (Edzes and Samulski 1977; Edzes and Samulski 1978; Wolff and Balaban 1989). With proper experimental design, the higher the density of nearby macromolecules in the tissue, the lower the resulting conventional MRI signal would be. This class of technique is known as magnetization transfer (MT) imaging.
</p>

<p style="text-align:justify;">
In the simplest and most used MT imaging method, only two images are acquired (one with MT preparation, and one without), and a normalized difference between the two images is calculated. This quantity is known as the magnetization transfer ratio (MTR), and has been used extensively to infer information on myelin diseases and disorders, such as multiple sclerosis. The proportional relationship between MTR and myelin density has been established using post-mortem immunohistological studies in humans (Schmierer et al. 2004; Schmierer et al. 2007) and animals (Merkler et al. 2005; Zaaraoui et al. 2008). MTR has also already been used in clinical drug trials for MS (Maguire et al. 2013; Brown et al. 2016). MTR has been widely used thanks to the fact that most scanners are equipped with the necessary software so that it can be added to an imaging protocol with the click of a button, and it is also a very quick measurement with a short acquisition time.
</p>
    
<p style="text-align:justify;">
In parallel to the exploration of the benefits of MTR for probing myelin density in disease and clinical applications, MR physicists developed other MT-related techniques that aim to extract quantitative physical information about the tissue using the mathematical models that describe the MT process. This sub-field is called quantitative MT, and the information that are extracted are the following MRI parameters: the pool-size ratio F (density of the macromolecular content’s (restricted pool) equilibrium magnetization divided by the the same value for the liquid content (free pool)), the exchange rate R, the longitudinal relaxation of the free pool T1f, and the transverse relaxation of both the free and restricted pools (T2f and T2r). However, quantitative MT has not been as widely used as MTR; because of the additional extracted parameters, additional measurements are needed which result in long imaging times unacceptable for clinical use. qMT also requires additional calibration measurements (B0, B1+, and T1), which further and can contribute to additional propagation of errors to the qMT parameters (Boudreau et al. 2018; Boudreau and Pike 2018). Despite this, qMT is a promising method as the measured parameters are desensitized to effects that can and do bias MTR measurements (eg T1, B1+). Another semi-quantitative technique that was recently developed with the aim of getting an MT measure that’s unbiased to T1 effects is the MT saturation (MTsat) technique, which will be the focus of Part 2 of this blog post series.
</p>
    
<p style="text-align:justify;">
We’ve recently published a blog post explaining the basics of qMT: the Bloch-Mconnell mathematical model of cross-relaxation, signal modelling, and signal fitting, presented with interactive figures generated using qMRLab. With that foundation of the fundamentals of MT, here we roll things back a bit and will explore MTR (Part 1) and MTsat (Part 2), the other two major applications of MT currently being used by the MRI community and which are much more in reach for most researchers to get setup and use. We’ll use the qMT simulation framework from qMRLab to demonstrate different properties and characteristics of MTR and MTsat, with the hopes that this will deepen the understanding of their benefits as well as their limitations.
</p>
</div>

<center> <h2 style="font-family:timesnewroman;font-size:30px">MTR: In Theory</h2> </center>

<div class=blog_body>
<p style="text-align:justify;">
The full mathematical description of the magnetization transfer two-pool exchange model was explained in our previous blog post on qMT. Although it’s these same equations that explain the signal differences between the two images acquired used to calculate MTR, in this section we’ll present a more conceptual explanation of the MT exchange process. In its most basic form, MT is modeled as an exchange process between two “pools” of protons, those from “mobile” protons named the “free” pool (those that are directly measured with conventional MRI), and those from “restricted” protons (i.e. macromoleules) named the “restricted” pool (these cannot be measured directly with conventional MRI). Restricted pool protons cannot be measured directly because the restricted movement creates a more static local electromagnetic environment that doesn’t average out, and this results in a transverse relaxation T2r (signal decay) that is too short to provide measurable signal (T2r ~ μs << feasible TE). Another consequence of this short signal decay time is a broadening of the absorption lineshape in the frequency domain (eg. the range of “resonant” frequencies of that pool of protons). This is a known property of the Fourier Transform, and the phenomenon is isomorphic to the quantum mechanics uncertainty principle, if you’re familiar with it; similar to Δx⋅Δp ≥ constant in quantum mechanics means that if Δx increases Δp will decrease, we observe a similar relationship approximated to T2 ⋅ FWHM of the frequencies = constant such that if T2 decreases, the FWHM of the frequencies will increase. If T2 is very very short (such as the case for macromolecules), the range of resonant frequencies will be very wide. MT leverages this property by selectively exciting restricted protons far from the mobile proton resonance frequency (applying a pulse off-resonance), but where the energy will be absorbed by some of the protons in the restricted pool. This is the initial preparation of the MT experiment.
</p>
    
    
<p style="text-align:justify;">
But even though we can selectively excite restricted pool protons by using an off-resonance pulse, we still cannot directly measure the signal from these.
</p>
</div>

In [ ]:
%use octave

% MATLAB/Octave code to simulate data for Figure 1
% Due to the long processing time of qMT simulations, this code is not recommended for use in Octave, and instead should be run on MATLAB
% For the purpose of this notebook, the simulates have been run on MATLAB R2020b and archived in a mat file, which is then loaded in python in the next cell.
% If you'd like to re-run or modify this code, we suggest you copy it to a MATLAB session, uncommend it, then run it.
% Note that you'll need qMRLab in the parent folder of where you execute this code

% %%
% clear all
% close all
% clc
% 
%% MATLAB/OCTAVE CODE
% Adds qMRLab to the path of the environment
% cd ../qMRLab
% startup
% %% Load protocols
% 
% fname = 'configs/mtr-protocols.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% protocols = [val.brown2013.siemens val.brown2013.philips val.karakuzu2022.siemens1 val.karakuzu2022.ge1]
% protocol_names = ['Brown2013 Siemens', 'Brown2013 Philips', 'Karakuzu2022 Siemens 1', 'Karakuzu2022 GE1']
% 
% %% Load tissues
% 
% fname = 'configs/tissues.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% tissues = [val.sled2001.healthycorticalgreymatter val.sled2001.healthywhitematter val.sled2001.nawm val.sled2001.earlywmlesion val.sled2001.latewmlesion];
% tissue_names = ['Healthy Cortical GM', 'Healthy WM', 'NAWM', 'Early WM MS Lesion', 'Late WM MS Lesion']
% 
% %%
% 
% MTRs = zeros(length(protocols),length(tissues))
% 
% for ii=1:length(protocols)
%     protocol = protocols(ii)
%     fa = protocol.fa
%     tr = protocol.tr/1000
%     te = protocol.te/1000
%     offset = protocol.offset
%     mt_shape = protocol.mtshape
%     mt_duration = protocol.mtduration/1000
%     mt_angle = protocol.mtangle
% 
%     Model = qmt_spgr;
%     Model.Prot.MTdata.Mat = [mt_angle, offset];
%     Model.Prot.TimingTable.Mat(5) = tr ;
%     Model.Prot.TimingTable.Mat(1) = mt_duration;
%     Model.Prot.TimingTable.Mat(4) = Model.Prot.TimingTable.Mat(5) - (Model.Prot.TimingTable.Mat(1) + Model.Prot.TimingTable.Mat(2) + Model.Prot.TimingTable.Mat(3)) ;
%     Model.options.Readpulsealpha = fa;
%     Model.options.MT_Pulse_Shape = mt_shape
%     
%     for jj = 1:length(tissues)
%         params = tissues(jj)
%         params = params {1}
%         x = struct;
%         x.F = params.F.mean;
%         x.kr = params.kf.mean / x.F;
%         x.R1f = params.R1f.mean;
%         x.R1r = 1;
%         x.T2f = params.T2f.mean/1000;
%         x.T2r = params.T2r.mean/(10^6);
% 
%         Opt.SNR = 1000;
%         Opt.Method = 'Bloch sim';
%         Opt.ResetMz = false;
% 
%         [FitResult, Smodel, Mz0] = Model.Sim_Single_Voxel_Curve(x,Opt);
% 
%         MTRs(ii,jj)=1-Smodel
%     end
% end
%
% save("fig1.mat", "tissue_names", "protocol_names", "MTRs")

In [ ]:
%use python3

# Prepare Python environment

import scipy.io as sio

#Load either archived or generated plot variables
mat_contents = sio.loadmat("fig_1.mat")

tissue_names = mat_contents['tissue_names'].tolist()
tissues = [
    tissue_names[0][0][0],
    tissue_names[0][1][0],
    tissue_names[0][2][0],
    tissue_names[0][3][0],
    tissue_names[0][4][0]
    ]
    
protocol_names = mat_contents['protocol_names'].tolist()
protocols = [
    protocol_names[0][0][0],
    protocol_names[0][1][0],
    protocol_names[0][2][0],
    protocol_names[0][3][0],
    ]
    
signal_brownSiemens = mat_contents["MTRs"][0]
signal_brownPhilips = mat_contents["MTRs"][1]
signal_karakuzuSiemens = mat_contents["MTRs"][2]
signal_karakuzuGE = mat_contents["MTRs"][3]

In [ ]:
%use python3

# Plot Figure 1

# Module imports

import matplotlib.pyplot as plt
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML
# PYTHON CODE

init_notebook_mode(connected=True)
# The polling here is to ensure that plotly.js has already been loaded before
# setting display alignment in order to avoid a race condition.


brown_siemens = go.Scatter(
    x = tissues,
    y = signal_brownSiemens,
    name = protocols[0],
    text = 'N/A',
    hoverinfo = 'y'
)


brown_philips = go.Scatter(
    x = tissues,
    y = signal_brownPhilips,
    name = protocols[1],
    hoverinfo = 'y'
)


karakuzu_siemens = go.Scatter(
    x = tissues,
    y = signal_karakuzuSiemens,
    name = protocols[2],
    hoverinfo = 'y'
)


karakuzu_ge = go.Scatter(
    x = tissues,
    y = signal_karakuzuGE,
    name = protocols[3],
    hoverinfo = 'y'
)


data = [brown_siemens, brown_philips, karakuzu_siemens, karakuzu_ge]

layout = go.Layout(
    width=600,
    height=600,
    margin=go.layout.Margin(
        l=100,
        r=80,
        b=100,
        t=130,
    ),
    annotations=[
        dict(
            x=-0.15,
            y=0.50,
            showarrow=False,
            text='MTR',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=-90,
            xref='paper',
            yref='paper'
        ),
    ],
    xaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2
    ),
    legend=dict(
        x=0.25,
        y=1.3,
        traceorder='normal',
        font=dict(
            family='Times New Roman',
            size=12,
            color='#000'
        ),
        bordercolor='#000000',
        borderwidth=2
    )
)

fig = dict(data=data, layout=layout)

plot(fig, filename = 'fig1.html', config = config)
display(HTML('fig1.html'))

<center> <h2 style="font-family:timesnewroman;font-size:30px">MTR: In Practice</h2> </center>

<div class=figure_caption>
<center>
<b style="text-align:justify;">
Figure 2. Simplified pulse sequence diagram of an MTR imaging sequence. An off-resnonance and high powered MT-preparation pulse is followed by a spoiler gradient to destroy any transverse magnetization prior the application of the imaging sequence, in this case a spoiled gradient recalled echo (SPGR).
</center>
</div>

<p>
<center><img src="mtspgr_pulsesequence.png" style="width:500px;height:auto;"></center>

<div class=figure_caption>
<center>
<b style="text-align:justify;">
Figure 3. MTR vs Protocols vs Tissues
</center>
</div>

In [ ]:
# Figure 4

In [ ]:
%use octave
%
% MATLAB/Octave code to simulate data for Figure 1
% Due to the long processing time of qMT simulations, this code is not recommended for use in Octave, and instead should be run on MATLAB
% For the purpose of this notebook, the simulates have been run on MATLAB R2020b and archived in a mat file, which is then loaded in python in the next cell.
% If you'd like to re-run or modify this code, we suggest you copy it to a MATLAB session, uncommend it, then run it.
% Note that you'll need qMRLab in the parent folder of where you execute this code
%
% clear all, close all, clc
% 
% %% Load protocol
% 
% fname = 'configs/mtr-protocols.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% protocol = val.brown2013.philips
% 
% %% Load tissues
% 
% fname = 'configs/tissues.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% tissue = val.sled2001.healthywhitematter;
% 
% %% T1 range
% 
% T1_true = 1/tissue{1}.R1f.mean
% T1_min = T1_true*0.7
% T1_max = T1_true*1.3
% 
% T1_range = linspace(T1_min, T1_max, 21)
% 
% 
% %%
% 
% MTRs = zeros(1,length(T1_range))
% 
% for ii=1:length(T1_range)
%     fa = protocol.fa
%     tr = protocol.tr/1000
%     te = protocol.te/1000
%     offset = protocol.offset
%     mt_shape = protocol.mtshape
%     mt_duration = protocol.mtduration/1000
%     mt_angle = protocol.mtangle
% 
%     Model = qmt_spgr;
%     Model.Prot.MTdata.Mat = [mt_angle, offset];
%     Model.Prot.TimingTable.Mat(5) = tr ;
%     Model.Prot.TimingTable.Mat(1) = mt_duration;
%     Model.Prot.TimingTable.Mat(4) = Model.Prot.TimingTable.Mat(5) - (Model.Prot.TimingTable.Mat(1) + Model.Prot.TimingTable.Mat(2) + Model.Prot.TimingTable.Mat(3)) ;
%     Model.options.Readpulsealpha = fa;
%     Model.options.MT_Pulse_Shape = mt_shape
%     
%     params = tissue{1}
%     x = struct;
%     x.F = params.F.mean;
%     x.kr = params.kf.mean / x.F;
%     x.R1f = 1/T1_range(ii);
%     x.R1r = 1;
%     x.T2f = params.T2f.mean/1000;
%     x.T2r = params.T2r.mean/(10^6);
%     
%     Opt.SNR = 1000;
%     Opt.Method = 'Bloch sim';
%     Opt.ResetMz = false;
%     
%     [FitResult, Smodel, Mz0] = Model.Sim_Single_Voxel_Curve(x,Opt);
%     
%     MTRs(1,ii)=1-Smodel
% end
% 
% save("fig4.mat", "T1_range", "MTRs", "T1_true")
% 

In [ ]:
%use python3

# Prepare Python environment

import scipy.io as sio

#Load either archived or generated plot variables
mat_contents = sio.loadmat("fig4.mat")

MTRs = mat_contents["MTRs"][0]
T1f = mat_contents["T1_range"][0]
T1_true = mat_contents["T1_true"][0]

x_T1_true = np.ones(len(MTRs))*T1_true[0]
y_T1_true = np.linspace(MTRs[0]*0.95, MTRs[-1]*1.05, num=len(MTRs))


In [ ]:
%use python3

# Plot Figure 1

# Module imports

import matplotlib.pyplot as plt
import plotly as py
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML
# PYTHON CODE

init_notebook_mode(connected=True)
# The polling here is to ensure that plotly.js has already been loaded before
# setting display alignment in order to avoid a race condition.


data = [
    go.Scatter(
        x=T1f,
        y=MTRs,
        name = "Protocol: Brown 2013 (Philips)",
        text = 'N/A',
        hoverinfo = 'y'
        ),
    go.Scatter(
        x=x_T1_true,
        y=y_T1_true,
        name = "True T1f",
        text = 'N/A',
        hoverinfo = 'y',
        line = dict(shape = 'linear', color = 'rgb(0, 0, 0)', width = 2, dash = 'dash'),
        ),
        
]


layout = go.Layout(
    width=600,
    height=600,
    margin=go.layout.Margin(
        l=100,
        r=80,
        b=100,
        t=130,
    ),
    annotations=[
        dict(
            x=-0.15,
            y=0.50,
            showarrow=False,
            text='MTR',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=-90,
            xref='paper',
            yref='paper'
        ),
        dict(
            x=0.5,
            y=-0.15,
            showarrow=False,
            text='T1f (s)',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=0,
            xref='paper',
            yref='paper'
        ),
    ],
    xaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2,
        range = [0.36, 0.5]
    ),
    legend=dict(
        x=0.25,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='Times New Roman',
            size=12,
            color='#000'
        ),
        bordercolor='#000000',
        borderwidth=2
    )
)

fig = dict(data=data, layout=layout)


plot(fig, filename = 'fig4.html', config = config)
display(HTML('fig4.html'))

# Figure 5

In [ ]:
%use octave

% clear all, close all, clc
% 
% %% Load protocol
% 
% fname = 'configs/mtr-protocols.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% protocol = val.brown2013.philips
% 
% %% Load tissues
% 
% fname = 'configs/tissues.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% tissue = val.sled2001.healthywhitematter;
% 
% %% B1 range
% 
% B1_min = 0.7
% B1_max = 1.3
% 
% B1_range = linspace(B1_min, B1_max, 21)
% 
% 
% %%
% 
% 
% MTRs = zeros(1,length(B1_range))
% 
% for ii=1:length(B1_range)
%     fa = protocol.fa*B1_range(ii)
%     tr = protocol.tr/1000
%     te = protocol.te/1000
%     offset = protocol.offset
%     mt_shape = protocol.mtshape
%     mt_duration = protocol.mtduration/1000
%     mt_angle = protocol.mtangle*B1_range(ii)
% 
%     Model = qmt_spgr;
%     Model.Prot.MTdata.Mat = [mt_angle, offset];
%     Model.Prot.TimingTable.Mat(5) = tr ;
%     Model.Prot.TimingTable.Mat(1) = mt_duration;
%     Model.Prot.TimingTable.Mat(4) = Model.Prot.TimingTable.Mat(5) - (Model.Prot.TimingTable.Mat(1) + Model.Prot.TimingTable.Mat(2) + Model.Prot.TimingTable.Mat(3)) ;
%     Model.options.Readpulsealpha = fa;
%     Model.options.MT_Pulse_Shape = mt_shape
%     
%     params = tissue{1}
%     x = struct;
%     x.F = params.F.mean;
%     x.kr = params.kf.mean / x.F;
%     x.R1f = tissue{1}.R1f.mean;
%     x.R1r = 1;
%     x.T2f = params.T2f.mean/1000;
%     x.T2r = params.T2r.mean/(10^6);
%     
%     Opt.SNR = 1000;
%     Opt.Method = 'Bloch sim';
%     Opt.ResetMz = false;
%     
%     [FitResult, Smodel, Mz0] = Model.Sim_Single_Voxel_Curve(x,Opt);
%     
%     MTRs(1,ii)=1-Smodel
% end
% 
% save("fig5.mat", "B1_range", "MTRs")
% 
% %%


In [ ]:
%use python3

# Prepare Python environment

import scipy.io as sio

#Load either archived or generated plot variables
mat_contents = sio.loadmat("fig5.mat")

MTRs = mat_contents["MTRs"][0]
B1 = mat_contents["B1_range"][0]
B1_true = 1

x_B1_true = np.ones(len(MTRs))*B1_true
y_B1_true = np.linspace(MTRs[0]*0.95, MTRs[-1]*1.05, num=len(MTRs))


In [ ]:
%use python3

# Plot Figure 1

# Module imports

import matplotlib.pyplot as plt
import plotly as py
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML
# PYTHON CODE

init_notebook_mode(connected=True)
# The polling here is to ensure that plotly.js has already been loaded before
# setting display alignment in order to avoid a race condition.


data = [
    go.Scatter(
        x=B1,
        y=MTRs,
        name = "Protocol: Brown 2013 (Philips)",
        text = 'N/A',
        hoverinfo = 'y'
        ),
    go.Scatter(
        x=x_B1_true,
        y=y_B1_true,
        name = "True B1 (B1 = 1)",
        text = 'N/A',
        hoverinfo = 'y',
        line = dict(shape = 'linear', color = 'rgb(0, 0, 0)', width = 2, dash = 'dash'),
        ),
        
]


layout = go.Layout(
    width=600,
    height=600,
    margin=go.layout.Margin(
        l=100,
        r=80,
        b=100,
        t=130,
    ),
    annotations=[
        dict(
            x=-0.15,
            y=0.50,
            showarrow=False,
            text='MTR',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=-90,
            xref='paper',
            yref='paper'
        ),
        dict(
            x=0.5,
            y=-0.15,
            showarrow=False,
            text='B1',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=0,
            xref='paper',
            yref='paper'
        ),
    ],
    xaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2,
        range = [0.36, 0.5]
    ),
    legend=dict(
        x=0.25,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='Times New Roman',
            size=12,
            color='#000'
        ),
        bordercolor='#000000',
        borderwidth=2
    )
)

fig = dict(data=data, layout=layout)


plot(fig, filename = 'fig6.html', config = config)
display(HTML('fig6.html'))

# Figure 6

In [ ]:
%use octave

% clear all, close all, clc
% 
% %% Load protocol
% 
% fname = 'configs/mtr-protocols.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% protocol = val.brown2013.philips
% 
% %% Load tissues
% 
% fname = 'configs/tissues.json'; 
% fid = fopen(fname); 
% raw = fread(fid,inf); 
% str = char(raw'); 
% fclose(fid); 
% val = loadjson(str);
% 
% tissue = val.sled2001.healthywhitematter;
% 
% %% TR range
% 
% TR_min = 21
% TR_max = 91
% 
% TR_range = linspace(TR_min, TR_max, 21)
% 
% 
% %%
% 
% 
% MTRs = zeros(1,length(TR_range))
% 
% for ii=1:length(TR_range)
%     fa = protocol.fa
%     tr = TR_range(ii)/1000
%     te = protocol.te/1000
%     offset = protocol.offset
%     mt_shape = protocol.mtshape
%     mt_duration = protocol.mtduration/1000
%     mt_angle = protocol.mtangle
% 
%     Model = qmt_spgr;
%     Model.Prot.MTdata.Mat = [mt_angle, offset];
%     Model.Prot.TimingTable.Mat(5) = tr ;
%     Model.Prot.TimingTable.Mat(1) = mt_duration;
%     Model.Prot.TimingTable.Mat(4) = Model.Prot.TimingTable.Mat(5) - (Model.Prot.TimingTable.Mat(1) + Model.Prot.TimingTable.Mat(2) + Model.Prot.TimingTable.Mat(3)) ;
%     Model.options.Readpulsealpha = fa;
%     Model.options.MT_Pulse_Shape = mt_shape
%     
%     params = tissue{1}
%     x = struct;
%     x.F = params.F.mean;
%     x.kr = params.kf.mean / x.F;
%     x.R1f = tissue{1}.R1f.mean;
%     x.R1r = 1;
%     x.T2f = params.T2f.mean/1000;
%     x.T2r = params.T2r.mean/(10^6);
%     
%     Opt.SNR = 1000;
%     Opt.Method = 'Bloch sim';
%     Opt.ResetMz = false;
%     
%     [FitResult, Smodel, Mz0] = Model.Sim_Single_Voxel_Curve(x,Opt);
%     
%     MTRs(1,ii)=1-Smodel
% 
% end
% 
% trueTR = protocol.tr
% 
% save("fig6.mat", "TR_range", "MTRs", "trueTR")
% 
% %%


In [ ]:
%use python3

# Prepare Python environment

import scipy.io as sio

#Load either archived or generated plot variables
mat_contents = sio.loadmat("fig6.mat")

MTRs = mat_contents["MTRs"][0]
TR = mat_contents["TR_range"][0]
TR_true = mat_contents["trueTR"][0]

x_TR_true = np.ones(len(MTRs))*TR_true
y_TR_true = np.linspace(MTRs[-1]*0.95, MTRs[0]*1.05, num=len(MTRs))


In [ ]:
%use python3

# Plot Figure 1

# Module imports

import matplotlib.pyplot as plt
import plotly as py
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML
# PYTHON CODE

init_notebook_mode(connected=True)
# The polling here is to ensure that plotly.js has already been loaded before
# setting display alignment in order to avoid a race condition.


data = [
    go.Scatter(
        x=TR,
        y=MTRs,
        name = "Protocol: Brown 2013 (Philips)",
        text = 'N/A',
        hoverinfo = 'y'
        ),
    go.Scatter(
        x=x_TR_true,
        y=y_TR_true,
        name = "True TR",
        text = 'N/A',
        hoverinfo = 'y',
        line = dict(shape = 'linear', color = 'rgb(0, 0, 0)', width = 2, dash = 'dash'),
        ),
        
]


layout = go.Layout(
    width=600,
    height=600,
    margin=go.layout.Margin(
        l=100,
        r=80,
        b=100,
        t=130,
    ),
    annotations=[
        dict(
            x=-0.15,
            y=0.50,
            showarrow=False,
            text='MTR',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=-90,
            xref='paper',
            yref='paper'
        ),
        dict(
            x=0.5,
            y=-0.15,
            showarrow=False,
            text='TR (ms)',
            font=dict(
                family='Times New Roman',
                size=22
            ),
            textangle=0,
            xref='paper',
            yref='paper'
        ),
    ],
    xaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='rgb(169,169,169)',
        linecolor='black',
        linewidth=2,
        range = [0.36, 0.5]
    ),
    legend=dict(
        x=0.25,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='Times New Roman',
            size=12,
            color='#000'
        ),
        bordercolor='#000000',
        borderwidth=2
    )
)

fig = dict(data=data, layout=layout)


plot(fig, filename = 'fig7.html', config = config)
display(HTML('fig7.html'))

# Figure7

In [ ]:
%use python3

# Prepare Python environment

import scipy.io as sio

#Load either archived or generated plot variables
mat_contents = sio.loadmat("fig7.mat")

MTRs = mat_contents["MTRs"]
TR = mat_contents["TR_range"][0]
B1 = mat_contents["B1_range"][0]


In [ ]:
%use python3

# Plot Figure 1

# Module imports

import matplotlib.pyplot as plt
import plotly as py
import plotly.graph_objs as go
import numpy as np
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
config={'showLink': False, 'displayModeBar': False}

init_notebook_mode(connected=True)

from IPython.core.display import display, HTML
# PYTHON CODE

init_notebook_mode(connected=True)
# The polling here is to ensure that plotly.js has already been loaded before
# setting display alignment in order to avoid a race condition.


data = [
    go.Contour(
        z=MTRs,
        ),
        
]


layout = go.Layout(
    width=600,
    height=600,
    margin=go.layout.Margin(
        l=100,
        r=80,
        b=100,
        t=130,
    ),
    legend=dict(
        x=0.25,
        y=1.2,
        traceorder='normal',
        font=dict(
            family='Times New Roman',
            size=12,
            color='#000'
        ),
        bordercolor='#000000',
        borderwidth=2
    )
)

fig = dict(data=data, layout=layout)


plot(fig, filename = 'fig8.html', config = config)
display(HTML('fig8.html'))

<center> <h2 style="font-family:timesnewroman;font-size:30px">Works Cited</h2> </center>

<div class=biblio_body>
<p style="text-align:justify;">
Barral JK, Gudmundson E, Stikov N, et al. (2010) A robust methodology for in vivo T<sub>1</sub> mapping. <i>Magn. Reson. Med.</i> 64(4): 1057–1067.
</p>

<p style="text-align:justify;">
Drain LE (1949) A Direct Method of Measuring Nuclear Spin-Lattice Relaxation Times. <i>Proceedings of the Physical Society. Section A</i> 62(5): 301–306.
</p>

<p style="text-align:justify;">
Fukushima, E. & Roeder, S., 1981. <i>Experimental Pulse NMR. A Nuts and Bolts Approach</i>, Reading, Massachusetts : Addison-Wesley Publ. Comp., Inc.
</p>

<p style="text-align:justify;">
Gai, N.D. et al., 2013. Modified Look-Locker T<sub>1</sub> evaluation using Bloch simulations: human and phantom validation. <i>Magn. Reson. Med.</i>, 69(2), pp.329–336.
</p>

<p style="text-align:justify;">
Hahn, E.L., 1949. An Accurate Nuclear Magnetic Resonance Method for Measuring Spin-Lattice Relaxation Times. <i>Physics Review</i>, 76(1), pp.145–146.
</p>

<p style="text-align:justify;">
Look, D.C. & Locker, D.R., 1970. Time Saving in Measurement of NMR and EPR Relaxation Times. <i>The Review of scientific instruments</i>, 41(2), pp.250–251.
</p>

<p style="text-align:justify;">
Messroghli, D.R. et al., 2004. Modified Look-Locker inversion recovery (MOLLI) for high-resolution T<sub>1</sub> mapping of the heart. <i>Magn. Reson. Med.</i>, 52(1), pp.141–146.
</p>

<p style="text-align:justify;">
Piechnik, S.K. et al., 2010. Shortened Modified Look-Locker Inversion recovery (ShMOLLI) for clinical myocardial T<sub>1</sub>-mapping at 1.5 and 3 T within a 9 heartbeat breathhold. <i>J. Cardiovasc. Magn. Reson.</i>, 12, p.69.
</p>

<p style="text-align:justify;">
Pykett, I.L. et al., 1983. Measurement of spin-lattice relaxation times in nuclear magnetic resonance imaging. <i>Physics in medicine and biology</i>, 28(6), pp.723–729.
</p>

<p style="text-align:justify;">
Pykett, I.L. & Mansfield, P., 1978. A line scan image study of a tumorous rat leg by NMR. <i>Physics in medicine and biology</i>, 23(5), pp.961–967.
</p>

<p style="text-align:justify;">
Steen, R.G. et al., 1994. Precise and accurate measurement of proton T<sub>1</sub> in human brain in vivo: validation and preliminary clinical application. <i>J. Magn. Reson. Imaging</i>, 4(5), pp.681–691.
</p>

<p style="text-align:justify;">
Stikov, N. et al., 2015. On the accuracy of T<sub>1</sub> mapping: Searching for common ground. <i>Magn. Reson. Med.</i>, 73(2), pp.514–522.
</p>
</div>

In [ ]:
%use python3
# PYTHON CODE

display(HTML(
    '<style type="text/css">'
    '.output_subarea {'
        'display: block;'
        'margin-left: auto;'
        'margin-right: auto;'
    '}'
    '.blog_body {'
        'line-height: 2;'
        'font-family: timesnewroman;'
        'font-size: 18px;'
        'margin-left: 0px;'
        'margin-right: 0px;'
    '}'
    '.biblio_body {'
        'line-height: 1.5;'
        'font-family: timesnewroman;'
        'font-size: 18px;'
        'margin-left: 0px;'
        'margin-right: 0px;'
    '}'
    '.note_body {'
        'line-height: 1.25;'
        'font-family: timesnewroman;'
        'font-size: 18px;'
        'margin-left: 0px;'
        'margin-right: 0px;'
        'color: #696969'
    '}'
    '.figure_caption {'
        'line-height: 1.5;'
        'font-family: timesnewroman;'
        'font-size: 16px;'
        'margin-left: 0px;'
        'margin-right: 0px'
    '</style>'
))